# Exploratory Data Analysis: Mental Health & Twitter Graph

This notebook performs the initial exploratory data analysis on the mental health text and Twitter network datasets. The goal is to understand the distribution of emotions, text properties, and the structural characteristics of the social network graph before advancing to modeling.

In [0]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, count, desc

# Initialize Spark Session
spark = SparkSession.builder.appName("EDA_MentalHealth").getOrCreate()

# Dynamically construct the absolute workspace path
repo_path = os.getcwd()

# Load datasets from the Datasets directory
mental_health_path = f"file:{repo_path}/mental_health.csv"
twitter_graph_path = f"file:{repo_path}/twitter_graph.csv"

mental_health_df = spark.read.csv(
    mental_health_path, 
    header=True, 
    inferSchema=True, 
    multiLine=True, 
    escape='"'
)

twitter_graph_df = spark.read.csv(
    twitter_graph_path, 
    header=True, 
    inferSchema=True
)

### Emotional Category Distribution
Analyzing the class balance across the six psychological categories (Positive, Neutral, Anger, Depression, Sadness, and Anxiety) to identify potential class imbalances before training machine learning models.

In [0]:
emotion_distribution = mental_health_df.groupBy("Emotion").count().orderBy(col("count").desc())
emotion_distribution.show()

### Text Length Variability
Assessing the length of the text entries to understand the distribution of message sizes, which will inform our tokenization, padding strategies, and feature engineering for NLP.

In [0]:
mental_health_df = mental_health_df.withColumn("text_length", length(col("Text")))
mental_health_df.select("text_length").summary("count", "mean", "min", "max").show()

### Twitter Graph Network Structure
Evaluating the scale of the directed relationships and checking for missing or inconsistent account metadata. This provides baseline sizing for the subsequent GraphFrames implementation.

In [0]:
print(f"Total Network Edges: {twitter_graph_df.count()}")

twitter_graph_df.select("Source Followers", "Destination Followers").summary("count", "mean", "max").show()
twitter_graph_df.groupBy("Source Verified").count().show()